In [1]:
from utils.spark_session import createSpark
from pyspark.sql import functions as F

spark = createSpark()






:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-af7a50a4-4179-40c9-84f0-5ce95ad1f794;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#c

In [2]:
TOPIC = "crypto.prices"
CHECKPOINT_BRONZE_PATH = f"s3a://spark-checkpoints/bronze/{TOPIC}"
CHECKPOINT_SILVER_PATH = f"s3a://spark-checkpoints/silver/{TOPIC}"
CHECKPOINT_GOLD_PATH = f"s3a://spark-checkpoints/gold/{TOPIC}"
BRONZE_PATH = f"s3a://crypto-lake/bronze/{TOPIC}"
SILVER_PATH = f"s3a://crypto-lake/silver/{TOPIC}"
GOLD_PATH = f"s3a://crypto-lake/gold/{TOPIC}"


In [3]:
df = spark.read.parquet(SILVER_PATH)
silver_schema = df.schema


In [4]:
df.printSchema()

root
 |-- ingestion_ts: timestamp (nullable = true)
 |-- kafka_ts: timestamp (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- current_price: decimal(28,8) (nullable = true)
 |-- market_cap: decimal(28,2) (nullable = true)
 |-- market_cap_rank: integer (nullable = true)
 |-- fully_diluted_valuation: decimal(28,2) (nullable = true)
 |-- total_volume: decimal(28,2) (nullable = true)
 |-- high_24h: decimal(28,8) (nullable = true)
 |-- low_24h: decimal(28,8) (nullable = true)
 |-- price_change_24h: decimal(28,8) (nullable = true)
 |-- price_change_pct_24h: float (nullable = true)
 |-- market_cap_change_24h: decimal(28,2) (nullable = true)
 |-- market_cap_change_pct_24h: float (nullable = true)
 |-- circulating_supply: decimal(28,8) (nullable = true)
 |-- total_supply: decimal(28,8) (nullable = true)
 |-- max_supply: decimal(28,8) (nullable = true)
 |-- ath: d

In [5]:
coin_ids = [
    "avalanche-2",
    "stable-2",
    "world-liberty-financial",
    "superstate-short-duration-us-government-securities-fund-ustb",
    "blackrock-usd-institutional-digital-liquidity-fund",
    "whitebit",
    "pudgy-penguins",
    "official-trump",
    "bittensor",
    "ethereum-classic",
    "bitget-token",
    "united-stables",
    "hash-2",
    "morpho",
    "vechain",
    "nexo",
    "memecore",
    "chainlink",
    "aptos",
    "okb",
    "falcon-finance",
    "htx-dao",
    "just",
    "blockchain-capital",
    "ondo-us-dollar-yield",
    "quant-network",
    "bitcoin",
    "arbitrum",
    "crypto-com-chain",
    "bitcoin-cash",
    "algorand",
    "near",
    "pi-network",
    "solana",
    "global-dollar",
    "render-token",
    "kaspa",
    "worldcoin-wld",
    "aave",
    "polkadot",
    "bonk",
    "ripple-usd",
    "usdtb",
    "hyperliquid",
    "internet-computer",
    "rain",
    "ripple",
    "zcash",
    "the-open-network",
    "midnight-3",
    "paypal-usd",
    "tron",
    "gatechain-token",
    "ethereum",
    "dexe",
    "usd1-wlfi",
    "jupiter-exchange-solana",
    "mantle",
    "sui",
    "binancecoin",
    "ousg",
    "bfusd",
    "uniswap",
    "litecoin",
    "cosmos",
    "dogecoin",
    "eutbl",
    "pepe",
    "canton-network",
    "ethena",
    "aster-2",
    "pump-fun",
    "sky",
    "skyai",
    "gho",
    "hedera-hashgraph",
    "stellar",
    "cardano",
    "ondo-finance",
    "leo-token",
    "tether-gold",
    "usd-coin",
    "usual-usd",
    "ethena-usde",
    "filecoin",
    "pax-gold",
    "xdce-crowd-sale",
    "kucoin-shares",
    "dash",
    "ylds",
    "usdd",
    "siren-2",
    "polygon-ecosystem-token",
    "tether",
    "beldex",
    "monero",
    "flare-networks",
    "terra-luna",
    "usds",
    "shiba-inu",
    "hashnote-usyc",
    "dai",
    "figure-heloc",
    "megausd",
    "janus-henderson-anemoy-treasury-fund"
]

In [6]:
import datetime

now = datetime.datetime.now()
date_limitation = now - datetime.timedelta(days=7)

def aggregate_prices(df_prices, coin_ids: list, granularity: str, date_limitation: datetime.datetime):
    return (df_prices
        .withColumn("bucket_start", F.date_trunc(granularity, F.col("event_time")))
        .where(F.col("coin_id").isin(coin_ids))
        .where(F.col("bucket_start") > date_limitation)
        .groupBy("coin_id", "bucket_start")
        .agg(
            F.max("current_price").alias("max_price"),
            F.max(F.struct("event_time", "market_cap")).alias("_last"),
        )
        .select(
            "coin_id", "bucket_start", "max_price",
            F.col("_last.market_cap").alias("market_cap"),
        )
        .orderBy("coin_id", "bucket_start")
    )

aggregated = aggregate_prices(df, coin_ids, "hour", date_limitation)



In [7]:
(aggregated.write
    .format("parquet")
    .partitionBy("coin_id")
    .mode("overwrite")
    .save(GOLD_PATH))